# Callbacks: Before & After

콜백으로 특정 시점에 로깅·검증·응답 수정·`state` 갱신 등을 할 수 있습니다.

- 문서: [Callbacks](https://google.github.io/adk-docs/callbacks/)

예제는 **Gemini** + Google AI Studio **무료 API 키**를 사용합니다.

| 시점 | 용도 예시 |
|------|-----------|
| Before / After **Agent** | 이번 요청의 에이전트 처리 전·후 |
| Before / After **Model** | LLM 호출 직전·직후 |
| Before / After **Tool** | 도구 실행 전·후 |

정책 모듈화는 **Plugins**도 검토하세요.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# 콜백 데모도 실제로 모델을 호출하므로 Gemini 키 필요
load_dotenv(Path(".env").resolve())
load_dotenv(Path("..") / ".env")
assert os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY"), (
    ".env 에 GOOGLE_API_KEY 또는 GEMINI_API_KEY 를 설정하세요."
)

## Before / After Agent

- `None` 반환: 정상적으로 모델 호출 등 다음 단계 진행  
- `types.Content` 반환: 모델 없이 그 내용을 응답처럼 사용 가능

In [ ]:
from typing import Optional

from google.adk.agents.callback_context import CallbackContext
from google.genai import types


def log_before_agent(ctx: CallbackContext) -> Optional[types.Content]:
    """에이전트가 이번 사용자 요청을 처리하기 '직전'에 호출됩니다."""
    print(f"[before_agent] agent={ctx.agent_name} invocation={ctx.invocation_id}")
    # 세션 state에 카운터를 올려 두면 이후 턴에서도 유지(커밋 방식은 ADK가 처리)
    ctx.state["turns"] = ctx.state.get("turns", 0) + 1
    return None  # None이면 기본 파이프라인(모델 호출 등) 계속


def log_after_agent(ctx: CallbackContext) -> Optional[types.Content]:
    """에이전트가 이번 요청 처리를 끝낸 '직후' 호출됩니다."""
    print(f"[after_agent] 누적 turns(state)={ctx.state.get('turns')}")
    return None

## Before / After Model

- Before에서 `LlmResponse`를 반환하면 실제 API 호출을 건너뛸 수 있음  
- After에서 응답을 가공·대체 가능

In [ ]:
from typing import Optional

from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse


def log_before_model(
    ctx: CallbackContext, req: LlmRequest
) -> Optional[LlmResponse]:
    """LLM으로 나갈 요청(req)을 검사·수정하거나, 여기서 가짜 응답을 줄 수 있습니다."""
    print("[before_model] model=", getattr(req, "model", "?"))
    return None


def log_after_model(
    ctx: CallbackContext, resp: LlmResponse
) -> Optional[LlmResponse]:
    """모델 응답(resp)을 받은 뒤 로깅·필터링·대체 응답 반환에 씁니다."""
    print("[after_model] 응답 수신 완료")
    return None

## 에이전트에 등록 후 실행

`LlmAgent` 생성 시 콜백 인자로 넘기면 ADK가 자동 호출합니다.

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP = "callback_lesson"
USER = "student"
SESSION = "lab1"

agent = LlmAgent(
    model="gemini-2.0-flash",
    name="callback_demo_agent",
    instruction="한국어로 한 문장만 답하세요.",
    before_agent_callback=log_before_agent,
    after_agent_callback=log_after_agent,
    before_model_callback=log_before_model,
    after_model_callback=log_after_model,
)

session_service = InMemorySessionService()
await session_service.create_session(
    app_name=APP, user_id=USER, session_id=SESSION, state={}
)
runner = Runner(agent=agent, app_name=APP, session_service=session_service)

# 한 번의 invocation: before_agent → … → 모델 → … → after_agent 순으로 콜백이 불립니다.
msg = types.Content(role="user", parts=[types.Part(text="콜백 데모야. 인사만 해줘.")])
async for event in runner.run_async(
    user_id=USER, session_id=SESSION, new_message=msg
):
    if event.is_final_response() and event.content and event.content.parts:
        t = event.content.parts[0].text
        if t:
            print("모델 최종 응답:", t)

await session_service.delete_session(
    app_name=APP, user_id=USER, session_id=SESSION
)